# 🏆 Macro Mission — Solution Walkthrough

**Category:** Automation | **Difficulty:** 🟣 Master

This notebook demonstrates the data manipulation side of the Macro Mission challenge using Python.
It also provides the complete **VBA code** you would use in Excel to automate the report formatting.

> VBA code cannot be executed in a Jupyter Notebook, but it is fully documented below for use in Excel's VBA Editor (Alt+F11).

## Prerequisites

```bash
pip install pandas openpyxl
```

In [ ]:
import pandas as pd
import numpy as np

print('Libraries loaded!')

## Step 1 — Generate Multi-Sheet Report Data

We simulate three sheets of daily sales reports — the data that the Macro must format.

In [ ]:
np.random.seed(7)

def make_report(sheet_name, n=10):
    """Generate a sample sales report for one sheet."""
    return pd.DataFrame({
        'Date': pd.date_range('2026-03-01', periods=n, freq='D').strftime('%Y-%m-%d'),
        'Region': np.random.choice(['North', 'South', 'East', 'West'], n),
        'Product': np.random.choice(['Widget A', 'Widget B', 'Gadget X'], n),
        'Units': np.random.randint(10, 100, n),
        'Revenue': np.random.randint(500, 5000, n)
    })

sheets = {
    'Week1': make_report('Week1'),
    'Week2': make_report('Week2'),
    'Week3': make_report('Week3')
}

for name, df in sheets.items():
    print(f'Sheet: {name}')
    print(df.to_string(index=False))
    print()

## Step 2 — Add Summary Row (Python Equivalent of VBA Loop)

The Macro's job is to loop over every sheet and add a totals row at the bottom.
Here we do the equivalent in Python.

In [ ]:
def add_summary_row(df):
    """Add a TOTAL row — equivalent to what the VBA macro adds in Excel."""
    summary = {
        'Date': 'TOTAL',
        'Region': '',
        'Product': '',
        'Units': df['Units'].sum(),
        'Revenue': df['Revenue'].sum()
    }
    return pd.concat([df, pd.DataFrame([summary])], ignore_index=True)

# Apply to all sheets
processed_sheets = {}
for name, df in sheets.items():
    processed_sheets[name] = add_summary_row(df.copy())
    last_row = processed_sheets[name].tail(1)
    print(f'Sheet {name} — TOTAL row: Units={last_row["Units"].values[0]}, Revenue=${last_row["Revenue"].values[0]:,}')

## Step 3 — VBA Code Reference

Below is the complete VBA code for the Macro Mission. Copy this into the Excel VBA Editor (Alt+F11).

### Macro 1: Format All Sheets

```vba
Sub FormatAllSheets()
    ' Loop through every sheet and apply formatting
    Dim ws As Worksheet
    
    For Each ws In ThisWorkbook.Worksheets
        ' Skip the Summary sheet
        If ws.Name <> "Summary" Then
            
            ' Bold the header row
            ws.Rows(1).Font.Bold = True
            
            ' Auto-fit all columns
            ws.Cells.EntireColumn.AutoFit
            
            ' Apply borders to the used range
            With ws.UsedRange.Borders
                .LineStyle = xlContinuous
                .Weight = xlThin
            End With
            
            ' Freeze the top row
            ws.Activate
            ActiveWindow.FreezePanes = False
            ws.Rows(2).Select
            ActiveWindow.FreezePanes = True
            
        End If
    Next ws
    
    MsgBox "All sheets formatted!", vbInformation
End Sub
```

### Macro 2: Add Summary Rows

```vba
Sub AddSummaryRows()
    Dim ws As Worksheet
    Dim lastRow As Long
    
    For Each ws In ThisWorkbook.Worksheets
        If ws.Name <> "Summary" Then
            
            ' Find the last row with data
            lastRow = ws.Cells(ws.Rows.Count, 1).End(xlUp).Row
            
            ' Add TOTAL label
            ws.Cells(lastRow + 1, 1).Value = "TOTAL"
            ws.Cells(lastRow + 1, 1).Font.Bold = True
            
            ' Sum numeric columns (adjust column indices to match your data)
            ws.Cells(lastRow + 1, 4).Formula = "=SUM(D2:D" & lastRow & ")"
            ws.Cells(lastRow + 1, 5).Formula = "=SUM(E2:E" & lastRow & ")"
            
        End If
    Next ws
    
    MsgBox "Summary rows added!", vbInformation
End Sub
```

## Step 4 — Save the Processed Data to Excel

We use `openpyxl` to write the formatted sheets to an `.xlsx` file.

In [ ]:
output_path = '/tmp/macro_mission_output.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for name, df in processed_sheets.items():
        df.to_excel(writer, sheet_name=name, index=False)

print(f'Output saved to: {output_path}')
print('Open this file in Excel and apply the VBA macros above to see the formatting in action.')

## ✅ Challenge Complete!

You've successfully:
- ✅ Understood how to **record a Macro** with Excel's Macro Recorder
- ✅ Learned to open the **VBA Editor** (Alt+F11)
- ✅ Written a **VBA loop** that processes multiple sheets
- ✅ Added **conditional logic** to skip the Summary sheet
- ✅ Added **summary rows** with SUM formulas to each sheet

### 🔗 Try Next:
- **[Power Query Quest](../../advanced/power-query-quest/)** — Automate data import and transformation
- **[Dashboard Duel](../../data-analysis/dashboard-duel/)** — Build a visual dashboard from your processed data